# Torso Decompositions — GBDT Showcase (Google Colab)

**Adrian Ymeri · University of Prishtina · SpOC-3 Torso Decompositions**

This notebook reproduces the headline result of the thesis on a fresh machine:
the **gradient-boosted-decision-tree (GBDT) adaptive-construction heuristic**
contributes positive hypervolume on the dense `large-graph`, and ~0 on the
saturated sparse instances — the *controlled ablation* (`WITH` vs `WITHOUT` the
GBDT orderings, everything else held fixed).

**How to run:** `Runtime → Run all`. When prompted in Step&nbsp;2, upload the
`torso_project.zip` that came with this notebook.

> The orderings are *generated fresh here* at modest budgets, so absolute scores
> are a little below the thesis numbers (which used longer runs and a larger
> pooled portfolio). The **sign and pattern** of the GBDT contribution — the
> point of the experiment — reproduce regardless. Increase the budgets in
> Step&nbsp;4 to close the gap.

## 1 · Install dependencies
LightGBM + XGBoost are the GBDT backends; `cmaes` is optional (the project ships
a built-in separable-CMA-ES, so this works even if `cmaes` fails to install).

In [ ]:
!pip -q install lightgbm xgboost scipy cmaes 2>/dev/null
import numpy, lightgbm, xgboost, scipy
print("numpy", numpy.__version__, "| lightgbm", lightgbm.__version__,
      "| xgboost", xgboost.__version__, "| scipy", scipy.__version__)

## 2 · Upload & unpack the project
Run this cell, then choose `torso_project.zip` from your computer.

In [ ]:
import os, zipfile, glob
from google.colab import files
up = files.upload()                      # pick torso_project.zip
zname = next(k for k in up if k.endswith('.zip'))
with zipfile.ZipFile(zname) as z: z.extractall('.')
# locate the project root (the dir that contains core.py)
ROOT = next(os.path.dirname(p) for p in glob.glob('**/core.py', recursive=True))
os.chdir(ROOT); print('project root:', os.getcwd())
print('instances:', sorted(os.listdir('data')))

## 3 · Sanity check the scorer
Loads each instance and scores a min-degree ordering with the *official*
evaluator built into `core.py` — confirms the environment is wired up.

In [ ]:
import random, core
for prob in ['small-graph','medium-graph','large-graph']:
    n, adj = core.load_graph(core.graph_path('.', prob))
    ab = core.build_adj_bitsets(n, adj)
    p = core.min_degree_perm(n, ab, rng=random.Random(0))
    w, t = core.evaluate(p, 0, ab, n)
    print(f"{prob:<13} n={n:<6} edges={sum(map(len,adj))//2:<7} min-degree width@t=0 = {w}")
print("leaderboard targets:", core.LEADERBOARD_TARGETS)

## 4 · Generate the orderings (the experiment)
For each instance we build two pools of elimination orderings into
`submissions/<instance>/`:

1. **`cmaes`** — the continuous-encoding search (spectral-policy CMA-ES). This is
   the GBDT-free baseline.
2. **`gbdt`** — the GBDT *adaptive* constructor (LightGBM decides each
   elimination step from the live residual graph).

Budgets below are Colab-friendly. **Raise them** (e.g. large → 600+) to approach
the thesis scores. `large-graph` is the slow one.

In [ ]:
BUDGETS = {'small-graph': 60, 'medium-graph': 90, 'large-graph': 180}  # seconds each
BACKEND = 'lightgbm'   # 'lightgbm' | 'xgboost' | 'hist' | 'ridge'

import subprocess, sys, time
def sh(cmd):
    print('›', ' '.join(cmd)); t=time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-600:] or '', r.stderr[-300:] if r.returncode else '')
    print(f'  [{time.time()-t:.0f}s]\n'); return r

for prob, b in BUDGETS.items():
    sh([sys.executable, 'algorithms/continuous/cmaes_torso.py',
        '--problem', prob, '--budget', str(b), '--seed', '42', '--engine', 'builtin'])
    sh([sys.executable, 'algorithms/continuous/gbdt_torso.py',
        '--problem', prob, '--budget', str(b), '--mode', 'construct',
        '--backend', BACKEND, '--seed', '42'])

## 5 · Controlled ablation — the proof
Builds the top-20 portfolio **twice** per instance — once using every ordering,
once **excluding** the `gbdt*` orderings — holding everything else identical. The
difference is GBDT's *independent* contribution to the submitted score (more
negative = better; a positive "share" means GBDT helped).

In [ ]:
r = subprocess.run([sys.executable, 'tools/ablation_gbdt.py'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

## 6 · What to expect
- **`small` / `medium`** — at or near their structural plateau: GBDT share ≈ **0**.
- **`large`** (dense) — GBDT adaptive construction finds hypervolume the
  geometric search alone does not: a **positive** share that grows with density.

This is the thesis claim in miniature: boosted-tree learning helps *specifically*
where fill-in dynamics dominate. In the thesis (longer runs, larger pooled
portfolio, LightGBM) the large-graph contribution is **+2,524 HV**, and is robust
across boosting library (LightGBM = XGBoost) and training objective
(pointwise = LambdaMART, via the `--rank` flag).

**Try next:** set `BACKEND='xgboost'` in Step 4 and re-run — the ablation gap is
implementation-agnostic. Or raise the budgets to push the absolute scores toward
the leaderboard targets printed in Step 3.